<a href="https://colab.research.google.com/github/monicachet99/NLP-DL/blob/main/Q%26A_for_ipc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q sentence-transformers faiss-cpu kagglehub pandas transformers torch

import os
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("✅ Libraries imported.")

print("\nLoading embedding model (all-mpnet-base-v2)...")
embed_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
print("✅ Embedding model loaded.")

print("\nLoading LLM (flan-t5-large)...")
llm_name    = "google/flan-t5-large"
tokenizer   = AutoTokenizer.from_pretrained(llm_name)
llm_model   = AutoModelForSeq2SeqLM.from_pretrained(llm_name)
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_model   = llm_model.to(device)
llm_model.eval()
print(f"✅ LLM loaded on {device}.")

import kagglehub

path = kagglehub.dataset_download(
    "dev523/indian-penal-code-ipc-sections-information"
)
print("Dataset path:", path)

csv_file = next(
    os.path.join(path, f) for f in os.listdir(path) if f.endswith(".csv")
)
df = pd.read_csv(csv_file)
print(f"✅ Loaded {len(df)} IPC sections.\n")

def make_chunk(row):
    desc = str(row["Description"])[:400].strip()
    return (
        f"Section {row['Section']}: {row['Offense']}. "
        f"Punishment: {row['Punishment']}. {desc}"
    )

data_texts = [make_chunk(row) for _, row in df.iterrows()]
print("Sample chunk:\n", data_texts[0])
print(f"\nTotal chunks: {len(data_texts)}")

print("\nCreating embeddings for all IPC sections...")
print("(~1-2 min on CPU, ~20s on GPU)\n")

emb_matrix = embed_model.encode(
    data_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print(f"\n✅ Embeddings shape: {emb_matrix.shape}")

dim   = emb_matrix.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb_matrix)
print(f"✅ FAISS index built with {index.ntotal} vectors.")

def retrieve_sections(query: str, top_k: int = 4) -> list:
    q_emb = embed_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [
        {"text": data_texts[idx], "score": float(score)}
        for score, idx in zip(scores[0], indices[0])
    ]

def generate_answer(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding=True,
    ).to(device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def ask_ipc_chatbot(user_query: str) -> str:
    retrieved = retrieve_sections(user_query, top_k=4)
    context   = "\n\n".join([r["text"] for r in retrieved])

    prompt = (
        f"You are a legal expert on the Indian Penal Code.\n"
        f"Answer the question using ONLY the IPC sections below.\n"
        f"Mention the relevant section numbers in your answer.\n\n"
        f"IPC Sections:\n{context}\n\n"
        f"Question: {user_query}\n\n"
        f"Answer:"
    )

    answer  = generate_answer(prompt)
    sources = "\n".join(
        [f"  [{i+1}] (score {r['score']:.2f}) {r['text'][:100]}..."
         for i, r in enumerate(retrieved)]
    )
    return f"{answer}\n\n📚 Retrieved Sections:\n{sources}"

print("\n" + "=" * 60)
print("   🏛️  IPC Chatbot  |  Powered by HuggingFace (Free)")
print("   Type your legal question. Type 'exit' to quit.")
print("=" * 60)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "bye"):
        print("Chatbot: Goodbye! Stay informed about your legal rights.")
        break

    print("\n⏳ Thinking...\n")
    try:
        answer = ask_ipc_chatbot(user_input)
        print(f"Chatbot:\n{answer}")
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
!pip install -q gradio

import gradio as gr

def ipc_chat(user_message, chat_history):
    if not user_message.strip():
        return "", chat_history

    try:
        retrieved = retrieve_sections(user_message, top_k=4)
        context   = "\n\n".join([r["text"] for r in retrieved])

        prompt = (
            f"You are a legal expert on the Indian Penal Code.\n"
            f"Answer the question using ONLY the IPC sections below.\n"
            f"Mention the relevant section numbers in your answer.\n\n"
            f"IPC Sections:\n{context}\n\n"
            f"Question: {user_message}\n\nAnswer:"
        )

        answer = generate_answer(prompt)

        sources_text = "\n\n---\n📚 **Retrieved Sections:**\n"
        for i, r in enumerate(retrieved):
            sources_text += f"\n**[{i+1}]** *(score: {r['score']:.2f})* {r['text'][:180]}...\n"

        full_response = answer + sources_text

    except Exception as e:
        full_response = f"⚠️ Error: {str(e)}"

    chat_history.append((user_message, full_response))
    return "", chat_history


SAMPLES = [
    "What is the punishment for murder under IPC?",
    "Which section covers theft and what is the penalty?",
    "What does IPC say about self-defence?",
    "Explain IPC Section 420 on cheating.",
    "What are the sections related to dowry harassment?",
    "What is the difference between theft and robbery in IPC?",
]


with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.amber,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
        font_mono=[gr.themes.GoogleFont("DM Mono"), "monospace"],
    ),
    css="""
        body { background: #f5f3ef; }

        .gradio-container {
            max-width: 820px !important;
            margin: 0 auto !important;
            background: #f5f3ef !important;
        }

        #ipc-header {
            text-align: center;
            padding: 28px 20px 10px;
            border-bottom: 1px solid #e2ddd6;
            margin-bottom: 16px;
        }
        #ipc-header h1 {
            font-size: 26px !important;
            font-weight: 700 !important;
            color: #926d2a !important;
            letter-spacing: -0.5px;
            margin-bottom: 4px !important;
        }
        #ipc-header p {
            font-size: 13px;
            color: #8a8880;
        }

        #chatbox {
            border: 1px solid #e2ddd6 !important;
            border-radius: 16px !important;
            background: #ffffff !important;
        }
        #chatbox .message.user {
            background: #fdf6e8 !important;
            border: 1px solid #edd9a3 !important;
            color: #2d2a24 !important;
            border-radius: 14px 14px 4px 14px !important;
            font-size: 14px !important;
        }
        #chatbox .message.bot {
            background: #f9f7f4 !important;
            border: 1px solid #e2ddd6 !important;
            color: #2d2a24 !important;
            border-radius: 14px 14px 14px 4px !important;
            font-size: 14px !important;
        }

        #input-row {
            margin-top: 10px;
        }
        #msg-input textarea {
            background: #ffffff !important;
            border: 1px solid #ddd8d0 !important;
            color: #2d2a24 !important;
            border-radius: 12px !important;
            font-size: 14px !important;
            padding: 12px 16px !important;
        }
        #msg-input textarea:focus {
            border-color: #b8892a !important;
            box-shadow: 0 0 0 2px rgba(184,137,42,0.12) !important;
        }
        #send-btn {
            background: linear-gradient(135deg, #b8892a, #d4a84b) !important;
            border: none !important;
            border-radius: 12px !important;
            color: #ffffff !important;
            font-weight: 600 !important;
            font-size: 14px !important;
            min-width: 90px !important;
        }
        #send-btn:hover { opacity: 0.88 !important; }

        #clear-btn {
            background: #ffffff !important;
            border: 1px solid #ddd8d0 !important;
            border-radius: 12px !important;
            color: #8a8880 !important;
            font-size: 13px !important;
        }
        #clear-btn:hover {
            border-color: #b8892a !important;
            color: #b8892a !important;
        }

        #samples-label {
            font-size: 11px !important;
            color: #8a8880 !important;
            letter-spacing: 0.8px !important;
            text-transform: uppercase !important;
            margin: 16px 0 8px !important;
        }
        .sample-btn button {
            background: #ffffff !important;
            border: 1px solid #ddd8d0 !important;
            border-radius: 20px !important;
            color: #5a5650 !important;
            font-size: 12px !important;
            padding: 6px 14px !important;
            transition: all 0.2s !important;
        }
        .sample-btn button:hover {
            border-color: #b8892a !important;
            color: #926d2a !important;
            background: #fdf6e8 !important;
        }

        #footer {
            text-align: center;
            font-size: 11px;
            color: #b8b4ae;
            font-family: 'DM Mono', monospace;
            letter-spacing: 0.6px;
            padding: 16px 0 8px;
        }

        #status-badge {
            display: inline-flex;
            align-items: center;
            gap: 6px;
            background: #f0faf0;
            border: 1px solid #b8ddb8;
            border-radius: 20px;
            padding: 4px 12px;
            font-size: 11px;
            color: #2d7a2d;
            font-family: 'DM Mono', monospace;
            margin-top: 8px;
        }
    """,
    title="IPC Legal Assistant",
) as demo:

    gr.HTML("""
        <div id="ipc-header">
            <h1>⚖️ IPC Legal Assistant</h1>
            <p>Indian Penal Code · RAG Pipeline · all-mpnet-base-v2 + flan-t5-large</p>
            <div id="status-badge">
                <span style="width:7px;height:7px;background:#2d7a2d;border-radius:50%;display:inline-block;"></span>
                Model Ready · 444 Sections Indexed
            </div>
        </div>
    """)

    chatbot = gr.Chatbot(
        value=[],
        elem_id="chatbox",
        height=400,
        show_label=False,
        bubble_full_width=False,
        render_markdown=True,
    )

    with gr.Row(elem_id="input-row"):
        msg = gr.Textbox(
            placeholder="Ask about any IPC section, offence, or punishment...",
            show_label=False,
            elem_id="msg-input",
            scale=5,
            lines=1,
        )
        send_btn = gr.Button("Send ➤", elem_id="send-btn", scale=1)
        clear_btn = gr.Button("Clear", elem_id="clear-btn", scale=1)

    gr.HTML('<div id="samples-label">Try asking</div>')
    with gr.Row():
        for q in SAMPLES[:3]:
            btn = gr.Button(q, elem_classes="sample-btn")
            btn.click(lambda x=q: x, outputs=msg)
    with gr.Row():
        for q in SAMPLES[3:]:
            btn = gr.Button(q, elem_classes="sample-btn")
            btn.click(lambda x=q: x, outputs=msg)

    gr.HTML('<div id="footer">IPC CHATBOT · HUGGINGFACE · LOCAL INFERENCE · FAISS VECTOR SEARCH</div>')

    send_btn.click(
        fn=ipc_chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot],
    )
    msg.submit(
        fn=ipc_chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot],
    )
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.launch(
    share=True,
    debug=False,
    show_error=True,
)